# MGPUSim Trace Analysis Notebook

**GPU Architecture Performance Analysis** — This notebook provides comprehensive analysis of simulation traces produced by [MGPUSim](https://github.com/sarchlab/mgpusim), a multi-GPU cycle-accurate simulator for AMD GCN3 GPUs.

**Trace Schema:**
```
trace (ID String, ParentID String, Kind String, What String, Location String, StartTime Float64, EndTime Float64)
```

**Location Hierarchy:** `GPU[N] → GPU[N].SA[M] → GPU[N].SA[M].CU[K] → GPU[N].SA[M].CU[K].<unit>`

**Navigate sections:**
- [Section 0](#section-0) — Setup & Connection
- [Section 1](#section-1) — Workload Overview
- [Section 2](#section-2) — GPU-level Performance
- [Section 3](#section-3) — Memory Hierarchy Analysis
- [Section 4](#section-4) — Compute Unit Deep Dive
- [Section 5](#section-5) — Request Chain Analysis
- [Section 6](#section-6) — Inter-GPU Communication
- [Section 7](#section-7) — Performance Summary Dashboard


## Section 0 — Setup & Connection <a id="section-0"></a>

Install dependencies and configure the ClickHouse connection. All queries aggregate data in the database — no full table scans into memory.


In [ ]:
import subprocess, sys
pkgs = ["clickhouse-connect", "pandas", "numpy", "matplotlib",
        "seaborn", "plotly", "networkx", "kaleido"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("All packages installed.")


In [ ]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import networkx as nx
import clickhouse_connect

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (14, 5),
                     "axes.titlesize": 13, "axes.labelsize": 11})
print("Imports OK.")


In [ ]:
# ── ClickHouse connection config ─────────────────────────────────────────────
CH_HOST     = "localhost"
CH_PORT     = 8123
CH_USER     = "mahmoud"
CH_PASSWORD = "yourpassword"
CH_DATABASE = "akitasimd7a9s1gjk23i97fmegs0"

# Configurable time-bucketing (splits simulation into N equal buckets)
TIME_BUCKETS = 100

client = clickhouse_connect.get_client(
    host=CH_HOST, port=CH_PORT,
    username=CH_USER, password=CH_PASSWORD,
    database=CH_DATABASE
)
print("Connected. Server version:", client.server_version)


In [ ]:
# ── Helper: parse Location string ────────────────────────────────────────────
_GPU = re.compile(r'GPU\[(\d+)\]')
_SA  = re.compile(r'SA\[(\d+)\]')
_CU  = re.compile(r'CU\[(\d+)\]')
_ROB = re.compile(r'L1VROB\[(\d+)\]')

_COMP_PATTERNS = [
    ("L1VCache",    r'L1VCache'),
    ("L1VROB",      r'L1VROB'),
    ("L1VAddrTrans",r'L1VAddrTrans'),
    ("L1VTLB",      r'L1VTLB'),
    ("L1SCache",    r'L1SCache'),
    ("L1SROB",      r'L1SROB'),
    ("L1ICache",    r'L1ICache'),
    ("L1IROB",      r'L1IROB'),
    ("DRAM",        r'DRAM'),
    ("L2Cache",     r'L2Cache'),
    ("RDMA",        r'RDMA'),
    ("CU",          r'\.CU\['),
]

def parse_location(loc: str) -> dict:
    g = _GPU.search(loc); sa = _SA.search(loc); cu = _CU.search(loc)
    gpu_id = int(g.group(1))  if g  else None
    sa_id  = int(sa.group(1)) if sa else None
    cu_id  = int(cu.group(1)) if cu else None
    comp = "Unknown"
    for name, pat in _COMP_PATTERNS:
        if re.search(pat, loc):
            comp = name; break
    # Execution unit inside CU (last segment after final dot)
    if comp == "CU":
        tail = loc.rsplit(".", 1)[-1]
        if tail in {"VALU","Scalar","VMem","Branch","LDS","GDS","Special","fetch","decode"}:
            comp = tail
    return dict(gpu_id=gpu_id, sa_id=sa_id, cu_id=cu_id, component=comp)

# ── Helper: extract base request ID (strip @location and _req_out) ────────────
def base_id(id_str: str) -> str:
    return id_str.split("@")[0].replace("_req_out","").strip("_")

# ── Helper: thin wrapper for ClickHouse → DataFrame ───────────────────────────
def q(sql: str) -> pd.DataFrame:
    return client.query_df(sql)

# ── Confirm helpers ───────────────────────────────────────────────────────────
sample = parse_location("GPU[1].SA[0].CU[2].VALU")
print("parse_location test:", sample)
print("base_id test:", base_id("498453@GPU[1].SA[0].L1VROB[1]"))


In [ ]:
# ── Global simulation metadata ────────────────────────────────────────────────
meta = q("""
    SELECT
        min(StartTime)          AS t_min,
        max(EndTime)            AS t_max,
        max(EndTime)-min(StartTime) AS t_span,
        count()                 AS total_events
    FROM trace
""").iloc[0]

T_MIN   = float(meta.t_min)
T_MAX   = float(meta.t_max)
T_SPAN  = float(meta.t_span)
BUCKET  = T_SPAN / TIME_BUCKETS   # seconds per bucket

print(f"Simulation: {T_MIN:.6e}s → {T_MAX:.6e}s  (span={T_SPAN:.6e}s)")
print(f"Total events: {int(meta.total_events):,}   Bucket width: {BUCKET:.6e}s")


## Section 1 — Workload Overview <a id="section-1"></a>

Understand the shape of the simulation: how long it ran, which GPUs were active, and what types of events dominate the trace. Key concepts:

- **Load balance**: ideally all GPUs finish at the same time; large gaps indicate imbalance.
- **Event type mix**: a compute-heavy trace has mostly `inst` events; memory-heavy workloads show many `req_in`/`req_out` events.
- **Time coverage**: a GPU with low utilization has long idle gaps relative to total simulation time.


In [ ]:
# Chart 1 — Timeline overview: activity density per GPU per time bucket
# Query: count events per GPU per time bucket (aggregated in CH)
timeline_df = q(f"""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        floor((StartTime - {T_MIN}) / {BUCKET})   AS bucket,
        count()                                    AS events
    FROM trace
    WHERE gpu_id != ''
    GROUP BY gpu_id, bucket
    ORDER BY gpu_id, bucket
""")
timeline_df["gpu_id"] = pd.to_numeric(timeline_df["gpu_id"])
timeline_df["bucket"] = pd.to_numeric(timeline_df["bucket"])

pivot = timeline_df.pivot_table(index="gpu_id", columns="bucket",
                                values="events", fill_value=0)

fig, ax = plt.subplots(figsize=(16, max(4, len(pivot)*0.5 + 1)))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd",
               extent=[0, TIME_BUCKETS, pivot.index.max()+0.5, pivot.index.min()-0.5])
ax.set_yticks(pivot.index)
ax.set_yticklabels([f"GPU[{i}]" for i in pivot.index])
ax.set_xlabel(f"Time bucket (each = {BUCKET*1e9:.2f} ns)")
ax.set_ylabel("GPU")
ax.set_title("Chart 1 — Timeline Activity Overview (event density per GPU per time bucket)")
plt.colorbar(im, ax=ax, label="Event count")
plt.tight_layout()
plt.savefig("chart01_timeline.png", dpi=100, bbox_inches="tight")
plt.show()
print("Interpretation: Dark columns indicate high simulation activity; white gaps are idle periods or synchronization barriers.")


> **Chart 1**: Each row is a GPU; each column is a time bucket (~simulation_time/100). Color intensity shows how many trace events fall in that bucket. Bright horizontal bands indicate active GPUs; vertical white stripes suggest global synchronization points (e.g., kernel launches, barriers).

In [ ]:
# Chart 2 — Event Kind distribution
kind_df = q("""
    SELECT Kind, count() AS cnt
    FROM trace
    GROUP BY Kind
    ORDER BY cnt DESC
""")

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(kind_df["Kind"], kind_df["cnt"] / 1e6,
              color=sns.color_palette("Set2", len(kind_df)))
ax.set_xlabel("Event Kind")
ax.set_ylabel("Count (millions)")
ax.set_title("Chart 2 — Event Kind Distribution")
for bar, val in zip(bars, kind_df["cnt"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val/1e6:.2f}M", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("chart02_kind_dist.png", dpi=100, bbox_inches="tight")
plt.show()
print("Kind counts:")
print(kind_df.to_string(index=False))


> **Chart 2**: The ratio of `inst` vs `req_in`/`req_out` events is a key indicator: a **compute-bound** workload has far more instructions than memory requests; a **memory-bound** workload shows comparable or larger memory traffic. High `wavefront` events suggest frequent wavefront dispatching (many small kernels or high occupancy).

In [ ]:
# Chart 3 — What (operation type) distribution
what_df = q("""
    SELECT What, count() AS cnt
    FROM trace
    GROUP BY What
    ORDER BY cnt DESC
    LIMIT 25
""")

fig, ax = plt.subplots(figsize=(14, 5))
colors = sns.color_palette("tab20", len(what_df))
bars = ax.barh(what_df["What"][::-1], what_df["cnt"][::-1] / 1e6, color=colors[::-1])
ax.set_xlabel("Count (millions)")
ax.set_title("Chart 3 — Top-25 'What' Field Distribution (operation types)")
for bar, val in zip(bars, what_df["cnt"][::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{val/1e6:.2f}M", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("chart03_what_dist.png", dpi=100, bbox_inches="tight")
plt.show()


> **Chart 3**: Instruction mix reveals workload character. VALU-heavy = arithmetic-intensive (matmul, conv). VMem-heavy = memory-access-intensive (BFS, streaming). High Special/Branch counts suggest control-flow divergence, which can hurt GPU efficiency significantly.

In [ ]:
# Chart 4 — Per-GPU time coverage (min/max StartTime, utilization)
cov_df = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        min(StartTime)   AS t_start,
        max(EndTime)     AS t_end,
        count()          AS events,
        sum(EndTime - StartTime) AS busy_time
    FROM trace
    WHERE gpu_id != ''
    GROUP BY gpu_id
    ORDER BY gpu_id
""")
cov_df["gpu_id"]    = pd.to_numeric(cov_df["gpu_id"])
cov_df["t_start"]   = pd.to_numeric(cov_df["t_start"])
cov_df["t_end"]     = pd.to_numeric(cov_df["t_end"])
cov_df["busy_time"] = pd.to_numeric(cov_df["busy_time"])
cov_df["span"]      = cov_df["t_end"] - cov_df["t_start"]
cov_df["util_pct"]  = 100 * cov_df["busy_time"] / (cov_df["span"] * 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: span bar chart
ax = axes[0]
ax.barh(cov_df["gpu_id"], (cov_df["t_end"] - T_MIN)*1e9,
        left=(cov_df["t_start"] - T_MIN)*1e9,
        color="steelblue", alpha=0.8, height=0.6)
ax.set_xlabel("Time (ns)")
ax.set_ylabel("GPU ID")
ax.set_title("GPU Active Time Span")
ax.set_yticks(cov_df["gpu_id"])
ax.set_yticklabels([f"GPU[{i}]" for i in cov_df["gpu_id"]])

# Right: utilization
ax = axes[1]
bars = ax.barh(cov_df["gpu_id"], cov_df["util_pct"],
               color=sns.color_palette("RdYlGn", len(cov_df)), height=0.6)
ax.set_xlabel("Utilization % (busy time / active span)")
ax.set_title("GPU Busy-Time Utilization %")
ax.set_yticks(cov_df["gpu_id"])
ax.set_yticklabels([f"GPU[{i}]" for i in cov_df["gpu_id"]])
ax.axvline(cov_df["util_pct"].mean(), color="red", linestyle="--",
           label=f"Mean={cov_df['util_pct'].mean():.1f}%")
ax.legend()
for bar, val in zip(bars, cov_df["util_pct"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=8)

plt.suptitle("Chart 4 — GPU Simulation Time Coverage & Utilization", fontsize=13)
plt.tight_layout()
plt.savefig("chart04_gpu_coverage.png", dpi=100, bbox_inches="tight")
plt.show()
print(cov_df[["gpu_id","span","util_pct"]].to_string(index=False))


> **Chart 4**: The active span shows if GPUs start/finish at the same time (good load balance). Utilization % = cumulative busy time / active span — values below 50% suggest significant stalls (cache misses, barriers, memory latency).

## Section 2 — GPU-level Performance <a id="section-2"></a>

Evaluate each GPU as a whole: instruction throughput, memory traffic volume, and load balance across GPUs. **Load imbalance** is one of the biggest performance killers in multi-GPU workloads — even if individual GPUs are fast, the system is limited by the slowest GPU (Amdahl's Law).

Key GPU architecture concepts:
- **Instruction throughput**: effective IPC × frequency × CU count
- **Memory bandwidth**: constrained by GDDR bandwidth and cache hierarchy
- **Load balance**: coefficient of variation (CV = std/mean) across GPUs; CV < 5% is good, > 20% indicates serious imbalance


In [ ]:
# Chart 5 — Per-GPU instruction throughput (instructions per ns)
tput_df = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        count()                                    AS inst_count,
        max(EndTime) - min(StartTime)              AS active_ns
    FROM trace
    WHERE Kind = 'inst' AND gpu_id != ''
    GROUP BY gpu_id
    ORDER BY gpu_id
""")
tput_df["gpu_id"]     = pd.to_numeric(tput_df["gpu_id"])
tput_df["inst_count"] = pd.to_numeric(tput_df["inst_count"])
tput_df["active_ns"]  = pd.to_numeric(tput_df["active_ns"]) * 1e9
tput_df["throughput"] = tput_df["inst_count"] / tput_df["active_ns"]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(tput_df["gpu_id"], tput_df["throughput"],
              color=sns.color_palette("Blues_d", len(tput_df)))
mean_tp = tput_df["throughput"].mean()
ax.axhline(mean_tp, color="red", linestyle="--", label=f"Mean={mean_tp:.2f} inst/ns")
ax.set_xlabel("GPU ID")
ax.set_ylabel("Instruction Throughput (inst / ns)")
ax.set_title("Chart 5 — Per-GPU Instruction Throughput")
ax.set_xticks(tput_df["gpu_id"])
ax.set_xticklabels([f"GPU[{i}]" for i in tput_df["gpu_id"]], rotation=45)
ax.legend()
for bar, val in zip(bars, tput_df["throughput"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + mean_tp*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig("chart05_gpu_throughput.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"CV (std/mean) = {tput_df['throughput'].std()/mean_tp*100:.1f}%  — lower is better load balance")


> **Chart 5**: The red dashed line is the mean throughput. GPUs below the mean are underperforming. A high coefficient of variation (>10%) indicates that work is not evenly distributed — check the kernel dispatch strategy.

In [ ]:
# Chart 6 — Per-GPU memory request volume (Read vs Write)
mem_df = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        What,
        count() AS req_count
    FROM trace
    WHERE Kind = 'req_in'
      AND What IN ('*mem.ReadReq', '*mem.WriteReq')
      AND gpu_id != ''
    GROUP BY gpu_id, What
    ORDER BY gpu_id, What
""")
mem_df["gpu_id"]    = pd.to_numeric(mem_df["gpu_id"])
mem_df["req_count"] = pd.to_numeric(mem_df["req_count"])

pivot_mem = mem_df.pivot_table(index="gpu_id", columns="What",
                               values="req_count", fill_value=0)
pivot_mem.columns = [c.replace("*mem.","") for c in pivot_mem.columns]

fig, ax = plt.subplots(figsize=(14, 5))
bottom = np.zeros(len(pivot_mem))
colors = {"ReadReq": "steelblue", "WriteReq": "salmon"}
for col in pivot_mem.columns:
    vals = pivot_mem[col].values / 1e3
    ax.bar(pivot_mem.index, vals, bottom=bottom/1e3 if col != list(pivot_mem.columns)[0] else 0,
           label=col, color=colors.get(col, "gray"))
    if col != list(pivot_mem.columns)[0]:
        bottom += pivot_mem[col].values
    else:
        bottom = pivot_mem[col].values

ax.set_xlabel("GPU ID")
ax.set_ylabel("Memory Requests (thousands)")
ax.set_title("Chart 6 — Per-GPU Memory Request Volume (Read vs Write)")
ax.set_xticks(pivot_mem.index)
ax.set_xticklabels([f"GPU[{i}]" for i in pivot_mem.index], rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig("chart06_gpu_memvol.png", dpi=100, bbox_inches="tight")
plt.show()
print("Read/Write ratio per GPU:")
if "ReadReq" in pivot_mem.columns and "WriteReq" in pivot_mem.columns:
    print((pivot_mem["ReadReq"] / pivot_mem["WriteReq"].replace(0,1)).to_string())


> **Chart 6**: Stacked bars show the read/write mix per GPU. Write-heavy workloads stress the write-back cache hierarchy more. A GPU with significantly more requests than others may be processing a disproportionate share of data — indicating data locality or partitioning issues.

In [ ]:
# Chart 7 — GPU load balance: CV across multiple metrics
balance_df = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        countIf(Kind = 'inst')                     AS inst_count,
        countIf(Kind = 'req_in')                   AS req_count,
        sum(EndTime - StartTime)                   AS busy_time,
        max(EndTime) - min(StartTime)              AS active_span
    FROM trace
    WHERE gpu_id != ''
    GROUP BY gpu_id
    ORDER BY gpu_id
""")
for c in ["inst_count","req_count","busy_time","active_span"]:
    balance_df[c] = pd.to_numeric(balance_df[c])
balance_df["gpu_id"] = pd.to_numeric(balance_df["gpu_id"])
balance_df["util"] = balance_df["busy_time"] / balance_df["active_span"]

metrics = {"Instructions": "inst_count",
           "Mem Requests": "req_count",
           "Busy Time (s)": "busy_time",
           "Utilization": "util"}

cvs = {name: balance_df[col].std() / balance_df[col].mean() * 100
       for name, col in metrics.items()}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
cv_s = pd.Series(cvs)
colors_cv = ["green" if v < 5 else "orange" if v < 20 else "red" for v in cv_s]
bars = ax.bar(cv_s.index, cv_s.values, color=colors_cv)
ax.axhline(5,  color="green",  linestyle="--", alpha=0.5, label="5% threshold (good)")
ax.axhline(20, color="orange", linestyle="--", alpha=0.5, label="20% threshold (warn)")
ax.set_ylabel("Coefficient of Variation (%)")
ax.set_title("Load Balance: CV per Metric (lower = more balanced)")
ax.legend()
for bar, val in zip(bars, cv_s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
            f"{val:.1f}%", ha="center", va="bottom")

ax = axes[1]
norm_df = balance_df.set_index("gpu_id")[list(metrics.values())].copy()
norm_df = (norm_df - norm_df.min()) / (norm_df.max() - norm_df.min() + 1e-12)
norm_df.columns = list(metrics.keys())
norm_df.T.plot(kind="bar", ax=ax, colormap="tab10")
ax.set_ylabel("Normalized value (0=min, 1=max)")
ax.set_title("Normalized Metric Distribution per GPU")
ax.set_xticklabels(list(metrics.keys()), rotation=20)
ax.legend(title="GPU", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

plt.suptitle("Chart 7 — GPU Load Balance Analysis", fontsize=13)
plt.tight_layout()
plt.savefig("chart07_load_balance.png", dpi=100, bbox_inches="tight")
plt.show()
print("CV summary:", cvs)


> **Chart 7**: Left: CV < 5% (green) is excellent balance; 5–20% (orange) is acceptable; > 20% (red) needs attention. Right: the normalized bar chart shows which GPUs are outliers on each metric. A GPU consistently at the top or bottom across metrics is the bottleneck.

In [ ]:
# Chart 8 — GPU active time fraction (fraction of total sim time each GPU is active)
act_df = q(f"""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        (max(EndTime) - min(StartTime)) / {T_SPAN}  AS active_fraction
    FROM trace
    WHERE gpu_id != ''
    GROUP BY gpu_id
    ORDER BY gpu_id
""")
act_df["gpu_id"]          = pd.to_numeric(act_df["gpu_id"])
act_df["active_fraction"] = pd.to_numeric(act_df["active_fraction"])

fig, ax = plt.subplots(figsize=(14, 4))
colors = ["steelblue" if v > 0.9 else "orange" if v > 0.7 else "salmon"
          for v in act_df["active_fraction"]]
ax.bar(act_df["gpu_id"], act_df["active_fraction"] * 100, color=colors)
ax.axhline(100, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("GPU ID")
ax.set_ylabel("Active Fraction (%)")
ax.set_title("Chart 8 — GPU Active Time Fraction (% of total simulation time)")
ax.set_xticks(act_df["gpu_id"])
ax.set_xticklabels([f"GPU[{i}]" for i in act_df["gpu_id"]], rotation=45)
ax.set_ylim(0, 110)
for x, y in zip(act_df["gpu_id"], act_df["active_fraction"]):
    ax.text(x, y*100+0.5, f"{y*100:.1f}%", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig("chart08_active_fraction.png", dpi=100, bbox_inches="tight")
plt.show()


> **Chart 8**: A GPU active for only 70% of the total simulation time sits idle for 30% — typically waiting for work dispatch, synchronization, or inter-GPU data transfers. Blue = >90% (good), orange = 70–90%, red = <70% (bottleneck).

## Section 3 — Memory Hierarchy Analysis <a id="section-3"></a>

The memory hierarchy is the most common GPU performance bottleneck. MGPUSim models a full GCN3 hierarchy: **CU → L1V cache (per-CU, write-around) → L2 cache (banked, write-back) → GDDR DRAM**.

Key concepts:
- **Latency per hop**: L1 hit ~5 cycles, L2 hit ~100 cycles, DRAM ~500+ cycles. A high L1-miss rate multiplies effective memory latency dramatically.
- **Cache thrashing**: too many wavefronts with large working sets evict each other's lines (L1V is direct-mapped in GCN3).
- **Memory-bound detection**: if avg memory latency >> L1 hit latency, the workload is memory-bound and cache optimization is needed.
- **Read/Write ratio**: write-heavy workloads can cause more cache evictions and write-back traffic to DRAM.


In [ ]:
# Chart 9 — Memory request latency distribution per component level
lat_df = q("""
    SELECT
        multiIf(
            Location LIKE '%L1VROB%',      'L1VROB',
            Location LIKE '%L1VCache%',    'L1VCache',
            Location LIKE '%L1SCache%',    'L1SCache',
            Location LIKE '%L1ICache%',    'L1ICache',
            Location LIKE '%L2Cache%',     'L2Cache',
            Location LIKE '%DRAM%',        'DRAM',
            Location LIKE '%CU[%',         'CU',
            Location LIKE '%RDMA%',        'RDMA',
            'Other'
        )                          AS component,
        quantile(0.25)(EndTime - StartTime) * 1e9 AS p25,
        quantile(0.50)(EndTime - StartTime) * 1e9 AS p50,
        quantile(0.75)(EndTime - StartTime) * 1e9 AS p75,
        quantile(0.95)(EndTime - StartTime) * 1e9 AS p95,
        avg(EndTime - StartTime)  * 1e9 AS avg_lat,
        count()                           AS n
    FROM trace
    WHERE Kind IN ('req_in', 'req_out')
      AND component != 'Other'
    GROUP BY component
    HAVING n > 10
    ORDER BY avg_lat
""")
for c in ["p25","p50","p75","p95","avg_lat"]:
    lat_df[c] = pd.to_numeric(lat_df[c])
lat_df["n"] = pd.to_numeric(lat_df["n"])

# Box-plot style using p25/p50/p75/p95
fig, ax = plt.subplots(figsize=(14, 6))
comps = lat_df["component"].tolist()
y = range(len(comps))
ax.barh(y, lat_df["p75"] - lat_df["p25"], left=lat_df["p25"],
        height=0.5, color="steelblue", alpha=0.7, label="IQR (p25–p75)")
ax.scatter(lat_df["p50"], y, color="white", edgecolors="navy", zorder=5, s=60, label="Median")
ax.scatter(lat_df["p95"], y, color="red", marker="|", s=200, zorder=5, label="p95")
ax.scatter(lat_df["avg_lat"], y, color="orange", marker="D", s=50, zorder=5, label="Mean")
ax.set_yticks(list(y))
ax.set_yticklabels(comps)
ax.set_xlabel("Latency (ns)")
ax.set_title("Chart 9 — Memory Request Latency Distribution by Component Level")
ax.legend()
# Annotate counts
for i, (_, row) in enumerate(lat_df.iterrows()):
    ax.text(row["p95"]+0.5, i, f"n={int(row['n']):,}  med={row['p50']:.1f}ns", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("chart09_mem_latency.png", dpi=100, bbox_inches="tight")
plt.show()
print(lat_df[["component","p25","p50","p75","p95","avg_lat","n"]].to_string(index=False))


> **Chart 9**: The box-plot style shows latency spread. L1VROB → L2Cache → DRAM should show increasing median latency. If DRAM latency is much larger than L2, the workload has poor cache reuse. Wide IQR (long boxes) indicates high variance — some requests are fast (cache hits), others slow (misses).

In [ ]:
# Chart 10 — L1 cache pressure heatmap: requests entering L1VROB per SA over time
l1_df = q(f"""
    SELECT
        toInt32(extract(Location, 'SA\\[(\\d+)\\]'))  AS sa_id,
        toInt32(floor((StartTime - {T_MIN}) / {BUCKET}))  AS bucket,
        count()                                            AS req_count
    FROM trace
    WHERE Location LIKE '%L1VROB%'
      AND Kind = 'req_in'
      AND sa_id >= 0
    GROUP BY sa_id, bucket
    ORDER BY sa_id, bucket
""")
l1_df["sa_id"]    = pd.to_numeric(l1_df["sa_id"])
l1_df["bucket"]   = pd.to_numeric(l1_df["bucket"])
l1_df["req_count"]= pd.to_numeric(l1_df["req_count"])

pivot_l1 = l1_df.pivot_table(index="sa_id", columns="bucket",
                               values="req_count", fill_value=0)

fig, ax = plt.subplots(figsize=(16, max(5, len(pivot_l1)*0.35 + 1)))
im = ax.imshow(pivot_l1.values, aspect="auto", cmap="hot_r",
               extent=[0, TIME_BUCKETS, pivot_l1.index.max()+0.5, pivot_l1.index.min()-0.5])
ax.set_yticks(pivot_l1.index[::max(1, len(pivot_l1)//16)])
ax.set_yticklabels([f"SA[{i}]" for i in pivot_l1.index[::max(1, len(pivot_l1)//16)]])
ax.set_xlabel(f"Time bucket (each ≈ {BUCKET*1e9:.2f} ns)")
ax.set_ylabel("Shader Array (SA) Index")
ax.set_title("Chart 10 — L1 Vector Cache Pressure Heatmap (requests/bucket per SA)")
plt.colorbar(im, ax=ax, label="L1VROB req_in count")
plt.tight_layout()
plt.savefig("chart10_l1_pressure.png", dpi=100, bbox_inches="tight")
plt.show()
print("Interpretation: Bright regions = high L1 pressure; dark = idle SA. Persistent bright rows may indicate cache thrashing in those SAs.")


> **Chart 10**: Each row is a Shader Array; each column is a time bucket. Bright (hot) cells indicate high L1 vector cache pressure at that time. Uniformly bright heatmaps are healthy; horizontal stripes mean some SAs always get more traffic (data imbalance); vertical stripes are synchronization barriers.

In [ ]:
# Chart 11 — Memory request flow Sankey diagram (request volume between hierarchy levels)
flow_df = q("""
    SELECT
        multiIf(
            Location LIKE '%\\.CU[%',      'Compute Unit',
            Location LIKE '%L1VROB%',    'L1V Cache',
            Location LIKE '%L1VCache%',  'L1V Cache',
            Location LIKE '%L1SCache%',  'L1S Cache',
            Location LIKE '%L2Cache%',   'L2 Cache',
            Location LIKE '%DRAM%',      'DRAM',
            Location LIKE '%RDMA%',      'RDMA',
            'Other'
        )           AS level,
        Kind,
        count()     AS cnt
    FROM trace
    WHERE Kind IN ('req_in', 'req_out')
      AND level != 'Other'
    GROUP BY level, Kind
    ORDER BY level, Kind
""")
flow_df["cnt"] = pd.to_numeric(flow_df["cnt"])

# Build Sankey: define nodes and flows based on memory hierarchy order
hierarchy = ["Compute Unit", "L1V Cache", "L1S Cache", "L2 Cache", "DRAM", "RDMA"]
req_by_level = flow_df[flow_df["Kind"]=="req_in"].set_index("level")["cnt"].to_dict()

nodes = [n for n in hierarchy if n in req_by_level]
node_idx = {n: i for i, n in enumerate(nodes)}

sources, targets, values = [], [], []
for i in range(len(nodes)-1):
    src, tgt = nodes[i], nodes[i+1]
    if src in req_by_level and tgt in req_by_level:
        # Flow = min of requests entering tgt (not all CU requests reach DRAM)
        val = min(req_by_level[src], req_by_level[tgt])
        sources.append(node_idx[src])
        targets.append(node_idx[tgt])
        values.append(val)

fig_sankey = go.Figure(go.Sankey(
    node=dict(
        pad=20, thickness=20,
        label=nodes,
        color=px.colors.qualitative.Plotly[:len(nodes)]
    ),
    link=dict(source=sources, target=targets, value=values,
              color="rgba(100,160,220,0.4)")
))
fig_sankey.update_layout(
    title_text="Chart 11 — Memory Request Flow Sankey (volume through hierarchy)",
    font_size=12, height=450
)
fig_sankey.show()
fig_sankey.write_html("chart11_sankey.html")
print("Sankey saved as chart11_sankey.html")
print("Level request counts:", req_by_level)


> **Chart 11** (interactive): The width of each flow band represents the volume of memory requests passing between hierarchy levels. Thick bands at lower levels (L2→DRAM) indicate many cache misses propagating down. A thin CU→L1V band with a thick L1V→L2 band means the L1 is saturated and passing most requests downstream.

In [ ]:
# Chart 12 — Read/Write ratio heatmap per GPU × SA
rw_df = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        extract(Location, 'SA\\[(\\d+)\\]')  AS sa_id,
        countIf(What = '*mem.ReadReq')             AS reads,
        countIf(What = '*mem.WriteReq')            AS writes
    FROM trace
    WHERE Kind = 'req_in'
      AND What IN ('*mem.ReadReq', '*mem.WriteReq')
      AND gpu_id != '' AND sa_id != ''
    GROUP BY gpu_id, sa_id
    ORDER BY gpu_id, sa_id
""")
rw_df["gpu_id"] = pd.to_numeric(rw_df["gpu_id"])
rw_df["sa_id"]  = pd.to_numeric(rw_df["sa_id"])
rw_df["reads"]  = pd.to_numeric(rw_df["reads"])
rw_df["writes"] = pd.to_numeric(rw_df["writes"])
rw_df["rw_ratio"] = rw_df["reads"] / (rw_df["writes"] + 1)

gpus = sorted(rw_df["gpu_id"].unique())
n_gpus = len(gpus)
cols = min(4, n_gpus)
rows = (n_gpus + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*4), squeeze=False)
for idx, gpu in enumerate(gpus):
    ax = axes[idx // cols][idx % cols]
    sub = rw_df[rw_df["gpu_id"] == gpu]
    pivot_rw = sub.pivot_table(index="sa_id", columns="gpu_id",
                               values="rw_ratio", fill_value=0)
    # Just plot as bar since only one GPU column
    sa_vals = sub.set_index("sa_id")["rw_ratio"].sort_index()
    colors = ["steelblue" if v >= 1 else "salmon" for v in sa_vals]
    ax.bar(sa_vals.index, sa_vals.values, color=colors)
    ax.axhline(1.0, color="red", linestyle="--", alpha=0.6, linewidth=0.8)
    ax.set_title(f"GPU[{gpu}]")
    ax.set_xlabel("SA index")
    ax.set_ylabel("Read/Write ratio")

# Hide empty subplots
for idx in range(n_gpus, rows * cols):
    axes[idx // cols][idx % cols].set_visible(False)

plt.suptitle("Chart 12 — Read/Write Ratio per GPU × Shader Array (>1 = read-dominant, <1 = write-dominant)",
             fontsize=13)
plt.tight_layout()
plt.savefig("chart12_rw_ratio.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"Overall R/W ratio: {rw_df['reads'].sum() / (rw_df['writes'].sum()+1):.2f}")


> **Chart 12**: Read-dominant (blue) workloads benefit most from caching. Write-dominant (red) SAs generate more write-back traffic and can saturate the write buffer. High variation across SAs within the same GPU suggests uneven data access patterns or partition imbalance.

## Section 4 — Compute Unit Deep Dive <a id="section-4"></a>

The Compute Unit (CU) is the fundamental execution unit in AMD GCN3 GPUs. Each CU contains 4 SIMD units (each executing 16 work items), a scalar unit, a branch unit, an LDS (local memory), and a vector memory unit.

Key concepts to look for:
- **CU utilization imbalance**: If some CUs execute far more instructions than others, the work-group dispatcher is creating hot spots.
- **Instruction mix**: VALU-heavy = compute-bound; VMem-heavy = memory-bound; high Branch = divergent control flow.
- **Pipeline stalls**: A high ratio of pending memory requests to total time indicates the CU is stalled waiting for memory, a classic memory-bound symptom.
- **Instruction execution time**: Long tail distributions suggest some wavefronts are stalling (cache misses, register dependencies, barriers).


In [ ]:
# Chart 13 — CU utilization heatmap (instruction count per CU: SA × CU grid)
cu_util = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        toInt32(extract(Location, 'SA\\[(\\d+)\\]'))  AS sa_id,
        toInt32(extract(Location, 'CU\\[(\\d+)\\]'))  AS cu_id,
        count()                                    AS inst_count
    FROM trace
    WHERE Kind = 'inst'
      AND gpu_id != '' AND sa_id >= 0 AND cu_id >= 0
    GROUP BY gpu_id, sa_id, cu_id
    ORDER BY gpu_id, sa_id, cu_id
""")
cu_util["gpu_id"]     = pd.to_numeric(cu_util["gpu_id"])
cu_util["sa_id"]      = pd.to_numeric(cu_util["sa_id"])
cu_util["cu_id"]      = pd.to_numeric(cu_util["cu_id"])
cu_util["inst_count"] = pd.to_numeric(cu_util["inst_count"])

gpus = sorted(cu_util["gpu_id"].unique())
n_gpus = len(gpus)
cols = min(3, n_gpus)
rows = (n_gpus + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols*6, rows*5), squeeze=False)
for idx, gpu in enumerate(gpus):
    ax = axes[idx // cols][idx % cols]
    sub = cu_util[cu_util["gpu_id"] == gpu]
    pivot = sub.pivot_table(index="sa_id", columns="cu_id",
                            values="inst_count", fill_value=0)
    im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"SA{i}" for i in pivot.index], fontsize=7)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"CU{i}" for i in pivot.columns], fontsize=7)
    ax.set_title(f"GPU[{gpu}]")
    ax.set_xlabel("CU index")
    ax.set_ylabel("SA index")
    plt.colorbar(im, ax=ax, label="Instructions")

for idx in range(n_gpus, rows * cols):
    axes[idx // cols][idx % cols].set_visible(False)

plt.suptitle("Chart 13 — CU Utilization Heatmap (instruction count per CU)", fontsize=14)
plt.tight_layout()
plt.savefig("chart13_cu_heatmap.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"Overall CU CV: {cu_util['inst_count'].std()/cu_util['inst_count'].mean()*100:.1f}%")


> **Chart 13**: Each cell is one CU. Dark red = heavy use, yellow/white = light use. Uniform color across all CUs indicates excellent CU-level load balance. Bright rows or columns indicate that certain SAs or CU positions consistently get more work.

In [ ]:
# Chart 14 — Instruction type mix per CU (stacked bar)
# Sample top GPUs/SAs to keep query manageable
mix_df = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        extract(Location, 'SA\\[(\\d+)\\]')  AS sa_id,
        extract(Location, 'CU\\[(\\d+)\\]')  AS cu_id,
        What                                       AS inst_type,
        count()                                    AS cnt
    FROM trace
    WHERE Kind = 'inst'
      AND What IN ('VALU','Scalar','VMem','Branch','LDS','GDS','Special')
      AND gpu_id != '' AND sa_id != '' AND cu_id != ''
    GROUP BY gpu_id, sa_id, cu_id, inst_type
    ORDER BY gpu_id, sa_id, cu_id, inst_type
""")
mix_df["gpu_id"] = pd.to_numeric(mix_df["gpu_id"])
mix_df["sa_id"]  = pd.to_numeric(mix_df["sa_id"])
mix_df["cu_id"]  = pd.to_numeric(mix_df["cu_id"])
mix_df["cnt"]    = pd.to_numeric(mix_df["cnt"])

# Focus on first GPU for the bar chart
first_gpu = mix_df["gpu_id"].min()
sub = mix_df[mix_df["gpu_id"] == first_gpu].copy()
sub["cu_label"] = sub.apply(lambda r: f"SA{int(r.sa_id)}.CU{int(r.cu_id)}", axis=1)
pivot_mix = sub.pivot_table(index="cu_label", columns="inst_type",
                             values="cnt", fill_value=0)
pivot_mix = pivot_mix.div(pivot_mix.sum(axis=1), axis=0) * 100  # normalize to %

inst_colors = {"VALU":"#4C72B0","Scalar":"#DD8452","VMem":"#55A868",
               "Branch":"#C44E52","LDS":"#8172B2","GDS":"#937860","Special":"#DA8BC3"}

fig, ax = plt.subplots(figsize=(16, max(6, len(pivot_mix)*0.35+1)))
bottom = np.zeros(len(pivot_mix))
for col in pivot_mix.columns:
    color = inst_colors.get(col, "gray")
    ax.barh(range(len(pivot_mix)), pivot_mix[col], left=bottom,
            label=col, color=color, height=0.8)
    bottom += pivot_mix[col].values
ax.set_yticks(range(len(pivot_mix)))
ax.set_yticklabels(pivot_mix.index, fontsize=7)
ax.set_xlabel("Instruction share (%)")
ax.set_title(f"Chart 14 — Instruction Type Mix per CU (GPU[{first_gpu}])")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig("chart14_inst_mix.png", dpi=100, bbox_inches="tight")
plt.show()
print("Overall instruction mix (%):")
print((mix_df.groupby("inst_type")["cnt"].sum() / mix_df["cnt"].sum() * 100).round(2).to_string())


> **Chart 14**: The horizontal stacked bars show each CU's instruction type mix as percentages. A CU with >50% VMem is likely memory-bound. High Branch % suggests control-flow divergence, reducing effective SIMD utilization. Uniform mixes across CUs confirm consistent workload distribution.

In [ ]:
# Chart 15 — CU pipeline stall proxy: ratio of pending-mem-request time to active time
stall_df = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        extract(Location, 'SA\\[(\\d+)\\]')  AS sa_id,
        extract(Location, 'CU\\[(\\d+)\\]')  AS cu_id,
        sum(EndTime - StartTime)                   AS total_mem_time,
        count()                                    AS mem_req_count
    FROM trace
    WHERE Kind IN ('req_in', 'req_out')
      AND Location LIKE '%\\.CU[%'
      AND gpu_id != '' AND sa_id != '' AND cu_id != ''
    GROUP BY gpu_id, sa_id, cu_id
""")

inst_time_df = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        extract(Location, 'SA\\[(\\d+)\\]')  AS sa_id,
        extract(Location, 'CU\\[(\\d+)\\]')  AS cu_id,
        sum(EndTime - StartTime)                   AS total_inst_time
    FROM trace
    WHERE Kind = 'inst'
      AND gpu_id != '' AND sa_id != '' AND cu_id != ''
    GROUP BY gpu_id, sa_id, cu_id
""")

for df in [stall_df, inst_time_df]:
    for c in ["gpu_id","sa_id","cu_id"]:
        df[c] = pd.to_numeric(df[c])

merged = stall_df.merge(inst_time_df, on=["gpu_id","sa_id","cu_id"], how="inner")
merged["total_mem_time"]  = pd.to_numeric(merged["total_mem_time"])
merged["total_inst_time"] = pd.to_numeric(merged["total_inst_time"])
merged["stall_rate"] = merged["total_mem_time"] / (merged["total_inst_time"] + 1e-30)
merged["cu_label"] = merged.apply(
    lambda r: f"GPU{int(r.gpu_id)}.SA{int(r.sa_id)}.CU{int(r.cu_id)}", axis=1)
merged_sorted = merged.sort_values("stall_rate", ascending=False).head(30)

fig, ax = plt.subplots(figsize=(14, 7))
colors = ["red" if v > 0.8 else "orange" if v > 0.4 else "steelblue"
          for v in merged_sorted["stall_rate"]]
ax.barh(range(len(merged_sorted)), merged_sorted["stall_rate"], color=colors)
ax.set_yticks(range(len(merged_sorted)))
ax.set_yticklabels(merged_sorted["cu_label"], fontsize=8)
ax.axvline(0.4, color="orange", linestyle="--", alpha=0.7, label="40% stall")
ax.axvline(0.8, color="red",    linestyle="--", alpha=0.7, label="80% stall")
ax.set_xlabel("Stall Rate (mem-pending time / instruction time)")
ax.set_title("Chart 15 — Top-30 CU Pipeline Stall Proxy (higher = more memory-bound)")
ax.legend()
plt.tight_layout()
plt.savefig("chart15_stall_rate.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"Mean stall rate: {merged['stall_rate'].mean()*100:.1f}%  Max: {merged['stall_rate'].max()*100:.1f}%")


> **Chart 15**: Stall rate = (time CU is processing memory requests) / (total instruction execution time). Values >40% (orange) indicate memory-bound CUs. Values >80% (red) are severely stalled — these CUs should be targeted first for optimization (prefetching, data layout changes, reducing working set).

In [ ]:
# Chart 16 — Instruction execution time distribution per instruction type
exec_df = q("""
    SELECT
        What                                           AS inst_type,
        quantile(0.10)(EndTime - StartTime) * 1e9  AS p10,
        quantile(0.25)(EndTime - StartTime) * 1e9  AS p25,
        quantile(0.50)(EndTime - StartTime) * 1e9  AS p50,
        quantile(0.75)(EndTime - StartTime) * 1e9  AS p75,
        quantile(0.90)(EndTime - StartTime) * 1e9  AS p90,
        quantile(0.99)(EndTime - StartTime) * 1e9  AS p99,
        avg(EndTime - StartTime) * 1e9             AS avg_lat,
        count()                                        AS n
    FROM trace
    WHERE Kind = 'inst'
      AND What IN ('VALU','Scalar','VMem','Branch','LDS','GDS','Special')
    GROUP BY inst_type
    HAVING n > 0
    ORDER BY avg_lat DESC
""")
for c in ["p10","p25","p50","p75","p90","p99","avg_lat"]:
    exec_df[c] = pd.to_numeric(exec_df[c])
exec_df["n"] = pd.to_numeric(exec_df["n"])

fig, ax = plt.subplots(figsize=(14, 6))
colors = [inst_colors.get(t, "gray") for t in exec_df["inst_type"]]
y = range(len(exec_df))

for i, (_, row) in enumerate(exec_df.iterrows()):
    ax.plot([row.p10, row.p90], [i, i], color=colors[i], linewidth=2, alpha=0.4)
    ax.barh(i, row.p75 - row.p25, left=row.p25, height=0.5,
            color=colors[i], alpha=0.8)
    ax.scatter([row.p50], [i], color="white", edgecolors=colors[i],
               zorder=5, s=80)
    ax.scatter([row.avg_lat], [i], color=colors[i], marker="D", s=50, zorder=5)
    ax.text(row.p99 + 0.2, i, f"n={int(row.n):,}  p99={row.p99:.1f}ns",
            va="center", fontsize=8)

ax.set_yticks(list(y))
ax.set_yticklabels(exec_df["inst_type"])
ax.set_xlabel("Execution Time (ns)")
ax.set_title("Chart 16 — Instruction Execution Time Distribution by Type\n"
             "(bar=IQR, whisker=p10-p90, circle=median, diamond=mean)")
plt.tight_layout()
plt.savefig("chart16_inst_exec_time.png", dpi=100, bbox_inches="tight")
plt.show()
print(exec_df[["inst_type","p25","p50","p75","p99","avg_lat","n"]].to_string(index=False))


> **Chart 16**: VMem instructions should have much wider distributions (cache hit vs miss makes 10–100× latency difference). VALU should be tight and fast (deterministic pipeline). A fat-tailed distribution for VALU suggests register pressure or issue stalls.

## Section 5 — Request Chain / Dependency Analysis <a id="section-5"></a>

In MGPUSim, memory requests form chains: a CU generates a `WriteReq`, which becomes a child request at L1VROB, then L2, then DRAM. The `ID` and `ParentID` fields encode these chains.

**ID encoding:**
- Plain ID (e.g., `498453`): instruction or wavefront ID
- `498453@GPU[1].SA[0].L1VROB[1]`: request 498453 traced at L1VROB (ParentID tracks origin)
- `498453_req_out`: the outgoing request created from request 498453

**Why chain depth matters:** The total latency seen by a CU is the sum of latencies across all hops. A long critical path (deep chain with no overlap) directly extends execution time.


In [ ]:
# Chart 17 — Request chain depth: count of distinct IDs per base parent chain
# Approach: count how many trace events share the same 'root' parent
# We approximate chain depth by counting how many children each ParentID spawns

chain_df = q("""
    SELECT
        ParentID,
        count() AS children
    FROM trace
    WHERE ParentID != '' AND ParentID != ID
    GROUP BY ParentID
    HAVING children > 0
    ORDER BY children DESC
    LIMIT 500000
""")
chain_df["children"] = pd.to_numeric(chain_df["children"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.hist(chain_df["children"], bins=50, color="steelblue", edgecolor="white",
        log=True, rwidth=0.85)
ax.set_xlabel("Number of children per parent ID")
ax.set_ylabel("Count (log scale)")
ax.set_title("Chart 17a — Children per Parent Request (log scale)")

ax = axes[1]
pct = chain_df["children"].value_counts().sort_index()
pct_cum = pct.cumsum() / pct.sum() * 100
ax.plot(pct_cum.index, pct_cum.values, color="steelblue", linewidth=2)
ax.axhline(90, color="red", linestyle="--", alpha=0.6, label="90th percentile")
ax.set_xlabel("Number of children")
ax.set_ylabel("Cumulative % of parents")
ax.set_title("Chart 17b — Cumulative Distribution of Chain Fan-out")
ax.legend()

plt.suptitle("Chart 17 — Request Chain Depth / Fan-out Distribution", fontsize=13)
plt.tight_layout()
plt.savefig("chart17_chain_depth.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"Chain stats: mean children={chain_df['children'].mean():.2f}, "
      f"max={chain_df['children'].max()}, "
      f"median={chain_df['children'].median():.0f}")


> **Chart 17**: Most requests should have 1–3 children (one per hierarchy level). A parent with many children indicates a coalescing unit that fans out into many sub-requests. A heavy tail means some requests spawn many sub-requests — possibly large DMA transfers or page migration events.

In [ ]:
# Chart 18 — Critical path: 10 longest request chains as waterfall charts
# Get the top-10 longest-running parent requests
longest = q("""
    SELECT
        ParentID,
        min(StartTime) AS chain_start,
        max(EndTime)   AS chain_end,
        max(EndTime) - min(StartTime) AS total_latency,
        count()        AS hops
    FROM trace
    WHERE ParentID != '' AND ParentID != ID
    GROUP BY ParentID
    ORDER BY total_latency DESC
    LIMIT 10
""")
longest["total_latency"] = pd.to_numeric(longest["total_latency"]) * 1e9
longest["chain_start"]   = pd.to_numeric(longest["chain_start"])
longest["chain_end"]     = pd.to_numeric(longest["chain_end"])
longest["hops"]          = pd.to_numeric(longest["hops"])

if len(longest) > 0:
    # For each of top-10, get all events in that chain
    top_ids = longest["ParentID"].tolist()
    # Build SQL IN clause
    ids_sql = "','".join(str(x) for x in top_ids)

    chain_events = q(f"""
        SELECT ID, ParentID, Kind, What, Location, StartTime, EndTime
        FROM trace
        WHERE ParentID IN ('{ids_sql}')
        ORDER BY ParentID, StartTime
        LIMIT 10000
    """)
    chain_events["StartTime"] = pd.to_numeric(chain_events["StartTime"])
    chain_events["EndTime"]   = pd.to_numeric(chain_events["EndTime"])

    fig, ax = plt.subplots(figsize=(16, 8))
    palette = sns.color_palette("tab10", 10)

    for i, (_, parent_row) in enumerate(longest.iterrows()):
        pid = parent_row["ParentID"]
        evts = chain_events[chain_events["ParentID"] == pid].copy()
        evts = evts.sort_values("StartTime")
        t0 = evts["StartTime"].min()
        y_base = i * 1.2

        for _, evt in evts.iterrows():
            start_ns = (evt["StartTime"] - t0) * 1e9
            dur_ns   = (evt["EndTime"] - evt["StartTime"]) * 1e9
            comp = evt["Location"].split(".")[-1][:12]
            ax.barh(y_base, dur_ns, left=start_ns, height=0.8,
                    color=palette[i], alpha=0.7, edgecolor="white")
            if dur_ns > parent_row["total_latency"] * 0.05:
                ax.text(start_ns + dur_ns/2, y_base, comp,
                        ha="center", va="center", fontsize=6, color="white")

        ax.text(-longest["total_latency"].max()*0.01, y_base,
                f"#{i+1} ({parent_row['total_latency']:.1f}ns, {int(parent_row['hops'])} hops)",
                va="center", ha="right", fontsize=8)

    ax.set_xlabel("Time from chain start (ns)")
    ax.set_yticks([])
    ax.set_title("Chart 18 — Top-10 Longest Request Chain Critical Paths (waterfall)")
    plt.tight_layout()
    plt.savefig("chart18_critical_path.png", dpi=100, bbox_inches="tight")
    plt.show()
else:
    print("No chain data found (no ParentID links in trace).")


> **Chart 18**: Each row is one long-latency request chain. Each bar segment is a child event (one hop in the hierarchy). The x-axis shows time from the chain's start. Long initial segments indicate early-stage bottlenecks; gaps between segments would reveal scheduling delays (though in this event-driven model, gaps are rare).

In [ ]:
# Chart 19 — Request fan-out: how many children each parent spawns
fanout_df = q("""
    SELECT
        children,
        count() AS parent_count
    FROM (
        SELECT ParentID, count() AS children
        FROM trace
        WHERE ParentID != '' AND ParentID != ID
        GROUP BY ParentID
    )
    GROUP BY children
    ORDER BY children
    LIMIT 200
""")
fanout_df["children"]     = pd.to_numeric(fanout_df["children"])
fanout_df["parent_count"] = pd.to_numeric(fanout_df["parent_count"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.bar(fanout_df["children"], fanout_df["parent_count"] / 1e3,
       color="steelblue", edgecolor="white", width=0.8)
ax.set_xlabel("Fan-out (number of children)")
ax.set_ylabel("Parent request count (thousands)")
ax.set_title("Fan-out Histogram")
ax.set_yscale("log")

ax = axes[1]
fanout_df["weighted"] = fanout_df["children"] * fanout_df["parent_count"]
ax.bar(fanout_df["children"], fanout_df["weighted"] / fanout_df["weighted"].sum() * 100,
       color="salmon", edgecolor="white", width=0.8)
ax.set_xlabel("Fan-out (number of children)")
ax.set_ylabel("% of total child events")
ax.set_title("Weighted Contribution to Total Request Volume")

plt.suptitle("Chart 19 — Request Fan-out Distribution", fontsize=13)
plt.tight_layout()
plt.savefig("chart19_fanout.png", dpi=100, bbox_inches="tight")
plt.show()
total_parents = fanout_df["parent_count"].sum()
mean_fanout = (fanout_df["children"] * fanout_df["parent_count"]).sum() / total_parents
print(f"Mean fan-out: {mean_fanout:.2f}  Total parent requests: {total_parents:,}")


> **Chart 19**: Fan-out of 1 means each memory request generates one child (sequential hierarchy traversal). Fan-out >4 (per CU coalescing) suggests some requests are being split into multiple sub-requests — typical when accessing non-contiguous memory. High fan-out events dominate total request volume even if rare.

## Section 6 — Inter-GPU Communication <a id="section-6"></a>

In multi-GPU systems, data must sometimes move between GPUs (e.g., for shared memory workloads, page migration, or RDMA-based communication). MGPUSim models RDMA engines and PCIe-like interconnects.

**Key metrics:**
- **Cross-GPU traffic**: requests where the Location's GPU index differs from the origin GPU (identified via ParentID chain traversal)
- **Communication vs compute overlap**: ideally, GPU B starts computing while GPU A is still sending data — poor overlap means communication is on the critical path.


In [ ]:
# Chart 20 — Cross-GPU traffic: RDMA component activity per GPU
rdma_df = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        Kind,
        count()             AS event_count,
        sum(EndTime - StartTime) * 1e9 AS total_time_ns
    FROM trace
    WHERE Location LIKE '%RDMA%' AND gpu_id != ''
    GROUP BY gpu_id, Kind
    ORDER BY gpu_id, Kind
""")

if len(rdma_df) == 0:
    print("No RDMA events found — this trace appears to be single-GPU or no cross-GPU communication occurred.")
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, "No inter-GPU (RDMA) events in trace",
            transform=ax.transAxes, ha="center", va="center", fontsize=14,
            bbox=dict(boxstyle="round", facecolor="lightyellow"))
    ax.set_title("Chart 20 — Inter-GPU RDMA Traffic (no data)")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig("chart20_cross_gpu.png", dpi=100, bbox_inches="tight")
    plt.show()
else:
    rdma_df["gpu_id"]       = pd.to_numeric(rdma_df["gpu_id"])
    rdma_df["event_count"]  = pd.to_numeric(rdma_df["event_count"])
    rdma_df["total_time_ns"]= pd.to_numeric(rdma_df["total_time_ns"])

    pivot_rdma = rdma_df.pivot_table(index="gpu_id", columns="Kind",
                                      values="event_count", fill_value=0)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    pivot_rdma.plot(kind="bar", ax=ax, colormap="Set2")
    ax.set_xlabel("GPU ID")
    ax.set_ylabel("RDMA Event Count")
    ax.set_title("RDMA Events per GPU by Kind")
    ax.set_xticklabels([f"GPU[{i}]" for i in pivot_rdma.index], rotation=45)

    ax = axes[1]
    pivot_time = rdma_df.pivot_table(index="gpu_id", columns="Kind",
                                      values="total_time_ns", fill_value=0)
    pivot_time.plot(kind="bar", ax=ax, colormap="tab10")
    ax.set_xlabel("GPU ID")
    ax.set_ylabel("Total RDMA Time (ns)")
    ax.set_title("RDMA Time per GPU by Kind")
    ax.set_xticklabels([f"GPU[{i}]" for i in pivot_time.index], rotation=45)

    plt.suptitle("Chart 20 — Inter-GPU RDMA Communication", fontsize=13)
    plt.tight_layout()
    plt.savefig("chart20_cross_gpu.png", dpi=100, bbox_inches="tight")
    plt.show()
    print(rdma_df.to_string(index=False))


> **Chart 20**: RDMA events (`req_in`/`req_out` at RDMA components) represent cross-GPU message-passing. If one GPU has significantly more RDMA traffic, it's the communication hub — a potential bottleneck. Ideally RDMA traffic is balanced and small relative to local compute.

In [ ]:
# Chart 21 — Communication vs compute overlap per GPU over time
overlap_df = q(f"""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]')             AS gpu_id,
        toInt32(floor((StartTime - {T_MIN}) / {BUCKET}))     AS bucket,
        countIf(Kind = 'inst')                                 AS compute_events,
        countIf(Location LIKE '%RDMA%')                        AS comm_events
    FROM trace
    WHERE gpu_id != ''
    GROUP BY gpu_id, bucket
    ORDER BY gpu_id, bucket
""")
overlap_df["gpu_id"]         = pd.to_numeric(overlap_df["gpu_id"])
overlap_df["bucket"]         = pd.to_numeric(overlap_df["bucket"])
overlap_df["compute_events"] = pd.to_numeric(overlap_df["compute_events"])
overlap_df["comm_events"]    = pd.to_numeric(overlap_df["comm_events"])

gpus = sorted(overlap_df["gpu_id"].unique())
n_gpus = min(len(gpus), 6)  # show up to 6 GPUs
cols = min(3, n_gpus)
rows = (n_gpus + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols*6, rows*3), squeeze=False)
for idx, gpu in enumerate(gpus[:n_gpus]):
    ax = axes[idx // cols][idx % cols]
    sub = overlap_df[overlap_df["gpu_id"] == gpu].sort_values("bucket")
    x = sub["bucket"] * BUCKET * 1e9
    ax.fill_between(x, sub["compute_events"], alpha=0.6, color="steelblue", label="Compute")
    ax.fill_between(x, sub["comm_events"],    alpha=0.6, color="salmon",    label="RDMA comm")
    ax.set_title(f"GPU[{gpu}]")
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("Event count")
    if idx == 0:
        ax.legend(fontsize=8)

for idx in range(n_gpus, rows * cols):
    axes[idx // cols][idx % cols].set_visible(False)

plt.suptitle("Chart 21 — Compute vs Communication Overlap per GPU over Time", fontsize=13)
plt.tight_layout()
plt.savefig("chart21_overlap.png", dpi=100, bbox_inches="tight")
plt.show()
print("Interpretation: Overlapping compute (blue) and RDMA (red) regions show effective overlap — hiding communication latency. Non-overlapping regions mean communication is serialized with compute.")


> **Chart 21**: When blue (compute) and red (communication) regions overlap in time, the GPU is hiding communication latency behind computation — ideal behavior. Flat blue lines during red peaks indicate the GPU is stalled waiting for inter-GPU data — a sign that the communication protocol or data partitioning needs improvement.

## Section 7 — Performance Summary Dashboard <a id="section-7"></a>

A high-level view of the most important system-wide metrics, followed by a radar chart comparing individual GPUs across multiple performance dimensions.

Use this section to quickly characterize the workload:
- **Compute-bound**: high instruction count, low memory latency, high utilization
- **Memory-bound**: high memory requests, high latency, low CU utilization
- **Communication-bound**: high RDMA traffic, low compute overlap


In [ ]:
# Summary dashboard — 6 key metrics as large display numbers
summary = q("""
    SELECT
        countIf(Kind = 'inst')                             AS total_instructions,
        countIf(Kind = 'req_in')                           AS total_mem_requests,
        avg(if(Kind='req_in', EndTime-StartTime, NULL))*1e9  AS avg_mem_latency_ns,
        (max(EndTime) - min(StartTime)) * 1e9              AS sim_time_ns
    FROM trace
""").iloc[0]

# CU utilization: instructions / (GPUs × SAs × CUs × sim_time)
cu_count_df = q("""
    SELECT count(DISTINCT concat(
        extract(Location,'GPU\\[(\\d+)\\]'), '_',
        extract(Location,'SA\\[(\\d+)\\]'), '_',
        extract(Location,'CU\\[(\\d+)\\]')
    )) AS n_cus
    FROM trace
    WHERE Kind = 'inst'
    LIMIT 1
""")

n_cus = int(pd.to_numeric(cu_count_df["n_cus"].iloc[0]))
total_insts   = int(pd.to_numeric(summary["total_instructions"]))
total_mem_req = int(pd.to_numeric(summary["total_mem_requests"]))
avg_mem_lat   = float(pd.to_numeric(summary["avg_mem_latency_ns"]))
sim_ns        = float(pd.to_numeric(summary["sim_time_ns"]))

mem_bw = total_mem_req / max(sim_ns, 1)  # requests/ns

# Load imbalance CV from earlier (recompute)
li_df = q("""
    SELECT extract(Location,'GPU\\[(\\d+)\\]') AS gpu,
           count() AS inst
    FROM trace WHERE Kind='inst' AND gpu!=''
    GROUP BY gpu
""")
li_df["inst"] = pd.to_numeric(li_df["inst"])
cv_imbalance  = li_df["inst"].std() / li_df["inst"].mean() * 100 if len(li_df)>1 else 0

avg_cu_util_df = q("""
    SELECT
        concat(extract(Location,'GPU\\[(\\d+)\\]'),'.',
               extract(Location,'SA\\[(\\d+)\\]'), '.',
               extract(Location,'CU\\[(\\d+)\\]')) AS cu_key,
        sum(EndTime-StartTime) / (max(EndTime)-min(StartTime)) AS util
    FROM trace WHERE Kind='inst' AND cu_key!='..'
    GROUP BY cu_key HAVING util > 0
""")
avg_cu_util = pd.to_numeric(avg_cu_util_df["util"]).mean() * 100 if len(avg_cu_util_df) > 0 else 0

metrics_labels = [
    "Total\nInstructions", "Mem Requests", "Avg Mem\nLatency (ns)",
    "Mem BW\n(req/ns)", "Load Imbalance\nCV (%)", "Simulation\nTime (µs)"
]
metrics_values = [
    f"{total_insts/1e6:.1f}M", f"{total_mem_req/1e6:.2f}M",
    f"{avg_mem_lat:.1f}", f"{mem_bw:.3f}",
    f"{cv_imbalance:.1f}%", f"{sim_ns/1e3:.2f}"
]
metrics_colors = ["#4C72B0","#55A868","#C44E52","#8172B2","#CCB974","#64B5CD"]

fig, axes = plt.subplots(2, 3, figsize=(16, 6))
for ax, label, val, color in zip(axes.flat, metrics_labels, metrics_values, metrics_colors):
    ax.set_facecolor(color)
    ax.text(0.5, 0.55, val, transform=ax.transAxes, ha="center", va="center",
            fontsize=28, fontweight="bold", color="white")
    ax.text(0.5, 0.2, label, transform=ax.transAxes, ha="center", va="center",
            fontsize=11, color="white", alpha=0.9)
    ax.axis("off")

plt.suptitle("Chart 22 — Performance Summary Dashboard", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig("chart22_dashboard.png", dpi=100, bbox_inches="tight")
plt.show()

print("=" * 50)
print("PERFORMANCE SUMMARY")
print("=" * 50)
for label, val in zip(metrics_labels, metrics_values):
    print(f"  {label.replace(chr(10),' '):30s}: {val}")
print(f"  {'Avg CU Utilization':30s}: {avg_cu_util:.1f}%")
print(f"  {'Active CUs':30s}: {n_cus}")


> **Chart 22**: Six key metrics at a glance. High imbalance CV (>10%) means load balancing is an issue. High memory latency (>>5ns for L1, >>100ns for L2) combined with low utilization confirms the workload is memory-bound. Low simulation time with high instruction count indicates high throughput.

In [ ]:
# Chart 23 — Radar/Spider chart comparing GPUs across 5 dimensions
# Compute 5 normalized metrics per GPU
radar_df = q("""
    SELECT
        extract(Location, 'GPU\\[(\\d+)\\]') AS gpu_id,
        countIf(Kind='inst')                        AS instructions,
        countIf(Kind='req_in')                      AS mem_requests,
        avg(if(Kind='req_in', EndTime-StartTime, NULL))*1e9 AS avg_lat_ns,
        sum(EndTime-StartTime) / (max(EndTime)-min(StartTime)) AS utilization,
        countIf(Location LIKE '%RDMA%')             AS rdma_events
    FROM trace
    WHERE gpu_id != ''
    GROUP BY gpu_id
    ORDER BY gpu_id
""")
for c in ["instructions","mem_requests","avg_lat_ns","utilization","rdma_events"]:
    radar_df[c] = pd.to_numeric(radar_df[c])
radar_df["gpu_id"] = pd.to_numeric(radar_df["gpu_id"])

# Normalize to 0-1 (higher = better, so invert latency)
eps = 1e-12
def norm(s): return (s - s.min()) / (s.max() - s.min() + eps)

radar_df["n_throughput"] = norm(radar_df["instructions"])
radar_df["n_mem_bw"]     = norm(radar_df["mem_requests"])
radar_df["n_latency"]    = 1 - norm(radar_df["avg_lat_ns"])   # lower latency = better
radar_df["n_util"]       = norm(radar_df["utilization"])
radar_df["n_comm"]       = 1 - norm(radar_df["rdma_events"])  # lower RDMA = more self-sufficient

categories = ["Inst\nThroughput", "Mem\nBandwidth", "Low\nLatency",
              "CU\nUtilization", "Low\nComm"]
dims = ["n_throughput","n_mem_bw","n_latency","n_util","n_comm"]
N = len(categories)
angles = [n / float(N) * 2 * 3.14159 for n in range(N)]
angles += angles[:1]  # close the loop

colors = plt.cm.tab10(np.linspace(0, 1, len(radar_df)))

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, polar=True)

for (_, row), color in zip(radar_df.iterrows(), colors):
    values = [row[d] for d in dims] + [row[dims[0]]]
    ax.plot(angles, values, linewidth=1.5, linestyle="solid",
            label=f"GPU[{int(row.gpu_id)}]", color=color)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25","0.5","0.75","1.0"], size=8)
ax.set_title("Chart 23 — GPU Performance Radar\n(normalized; outer edge = best)",
             size=13, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=9)
plt.tight_layout()
plt.savefig("chart23_radar.png", dpi=100, bbox_inches="tight")
plt.show()
print("Radar normalization: 1.0 = best GPU on that metric, 0.0 = worst.")
print("GPUs closer to the outer edge on all axes are best-balanced performers.")


> **Chart 23**: The radar chart provides an at-a-glance multi-dimensional comparison of all GPUs. A GPU that covers most of the outer area on all axes is the highest-performing and most balanced. Consistent shapes across all GPUs mean good overall balance. A GPU that bulges on "Low Latency" but shrinks on "Instruction Throughput" is likely memory-bound.

---

## Analysis Complete

All charts have been saved as PNG/HTML files in the current directory. Key files:

| Chart | File | Section |
|-------|------|---------|
| Timeline activity | `chart01_timeline.png` | Workload Overview |
| Kind distribution | `chart02_kind_dist.png` | Workload Overview |
| What distribution | `chart03_what_dist.png` | Workload Overview |
| GPU time coverage | `chart04_gpu_coverage.png` | Workload Overview |
| GPU throughput | `chart05_gpu_throughput.png` | GPU Performance |
| Memory volume | `chart06_gpu_memvol.png` | GPU Performance |
| Load balance | `chart07_load_balance.png` | GPU Performance |
| Active fraction | `chart08_active_fraction.png` | GPU Performance |
| Mem latency | `chart09_mem_latency.png` | Memory Hierarchy |
| L1 pressure | `chart10_l1_pressure.png` | Memory Hierarchy |
| Flow Sankey | `chart11_sankey.html` | Memory Hierarchy |
| R/W ratio | `chart12_rw_ratio.png` | Memory Hierarchy |
| CU heatmap | `chart13_cu_heatmap.png` | CU Deep Dive |
| Inst mix | `chart14_inst_mix.png` | CU Deep Dive |
| Stall rate | `chart15_stall_rate.png` | CU Deep Dive |
| Inst exec time | `chart16_inst_exec_time.png` | CU Deep Dive |
| Chain depth | `chart17_chain_depth.png` | Request Chains |
| Critical path | `chart18_critical_path.png` | Request Chains |
| Fan-out | `chart19_fanout.png` | Request Chains |
| Cross-GPU | `chart20_cross_gpu.png` | Inter-GPU |
| Comm overlap | `chart21_overlap.png` | Inter-GPU |
| Dashboard | `chart22_dashboard.png` | Summary |
| Radar | `chart23_radar.png` | Summary |
